# Miner Tips 05: Diagnostics and Traps

This notebook is a guardrail. It shows how to compare recipes and avoid common false positives.

Good compression work tracks at least three things:

```text
artifact size
public/dev PPL
loadability/reproducibility
```

Bad compression work often optimizes one proxy and then fails text quality.

In [ ]:
import math

experiments = [
    {"name": "pure_binary_everywhere", "size_gb": 6.2, "public_ppl": 95.0, "loads": True},
    {"name": "binary_with_10pct_q4_rescue", "size_gb": 7.1, "public_ppl": 38.0, "loads": True},
    {"name": "layerwise_distilled_ternary", "size_gb": 9.4, "public_ppl": 27.0, "loads": True},
    {"name": "proxy_only_low_mse", "size_gb": 5.8, "public_ppl": 180.0, "loads": True},
    {"name": "broken_packaging", "size_gb": 7.0, "public_ppl": 30.0, "loads": False},
]

# Sort by public/dev PPL, but reject artifacts that do not load.
valid = [e for e in experiments if e["loads"] and math.isfinite(e["public_ppl"])]
for e in sorted(valid, key=lambda item: item["public_ppl"]):
    print(f"{e['name']:30s} size={e['size_gb']:4.1f} GB  public/dev PPL={e['public_ppl']:6.2f}")

In [ ]:
def is_dominated(candidate, others):
    """Return True if another experiment is both smaller and lower-PPL."""
    for other in others:
        if other is candidate:
            continue
        smaller_or_equal = other["size_gb"] <= candidate["size_gb"]
        better_or_equal = other["public_ppl"] <= candidate["public_ppl"]
        strictly_better = other["size_gb"] < candidate["size_gb"] or other["public_ppl"] < candidate["public_ppl"]
        if smaller_or_equal and better_or_equal and strictly_better:
            return True
    return False

frontier = [e for e in valid if not is_dominated(e, valid)]
print("frontier candidates:")
for e in sorted(frontier, key=lambda item: item["size_gb"]):
    print("-", e["name"])

## Traps to avoid

- **Pure low-bit everywhere**: fragile rows/modules can dominate PPL.
- **Proxy-only wins**: low reconstruction error does not always mean good text modeling.
- **Single-example tuning**: a recipe can memorize a tiny public/dev sample and fail elsewhere.
- **Broken artifacts**: validators need to download, verify, and load the artifact without your notebook.
- **Local-machine assumptions**: no absolute operator paths, caches, private files, or auth secrets.

## A better experiment report

```text
Target: binary or ternary competition
Base model: public model id or local public checkpoint
Compression recipe: rowmix / q4 rescue / layerwise distillation / widening
Public/dev data: describe source and token count
Public/dev PPL: number
Artifact size: bytes
SHA-256: exact digest
Known risks: what may fail and what you did not test
```